# Datamine CLUSTR Process Example

This notebook demonstrates how to configure and run the **`clustr`** process wrapper in `dmstudio`.

## Process Description

## CLUSTR

Cluster Analysis is a data exploration (mining) tool for dividing a multivariate dataset into natural clusters (groups). We use the methods to explore whether previously undefined clusters (groups) may exist in the dataset. For instance, a marketing department may wish to use survey results to sort its customers into categories (perhaps those likely to be most receptive to buying a product, those most likely to be against buying a product, and so forth).

Cluster Analysis is used when we believe that the sample units come from an unknown number of distinct populations or sub-populations. We also assume that the sample units come from a number of distinct populations, but there is no apriori definition of those populations. Our objective is to describe those populations using the observed data.

Fields are clustered (R-mode) into groups on the basis of the correlation matrix to simplify the structure. R - mode cluster analysis (agglomerative) using the similarity or dissimilarity matrix with interactive screen graphics to define and plot types of clustering:

[![](../Images/CLUSTR.jpg)](<javascript:void\(0\);>)

The results are presented as a dendrogram and a Kleiner - Hartigan tree (see below for an example). Variables can be selected for clustering using field and retrieval criteria.

### File Handling

The input data file (&IN) must have at least three numeric fields plus a sample identifier field (@**SAMPID**) which must be declared on input. There is also the optional facility of imputing and outputting the correlation matrix.

### Special Features

The user has the option of defining the type of correlation matrix to be used, i.e.. similarity (variance covariance) or dissimilarity (euclidean distance) with the interactive selection of type of cluster linkage model to be applied to the dendrogram. Note, if the similarity matrix is selected (@**MATTYPE** =0) then the default value for the Z transformation of the data (normalized data with the maximum value set to 1) must also be used (@**ZTRAN** =1). If the dissimilarity matrix is used then the Z transformation of the data is optional.

Dendrograms, are plotted from high to low using the similarity coefficients generated from the hierarchical structure. The dissimilarity (distance) matrix is scaled from low (close together) to high (farther away) using the euclidean distance. Ward's method can give high similarity values in excess of -1.0 and care should be taken not to relate the similarities too closely to the matrix correlation coefficients.

The type of hierarchical linkage required to generate the dendrogram (see Figure 1) can be defined interactively by the user including nearest neighbor, furthest neighbor, simple averages, median, group averages, centroid and Ward's method. Difficulties arise in order to select the method to use to give the 'best ' results. Mathematically nearest neighbor is considered the best method.

However as a measure of 'goodness of fit' between the observed data and the generated hierarchical structure, the cophonetic correlation coefficient is used. The cophonetic correlation coefficient is the correlation coefficient between the observed and the hierarchical clustering similarity coefficients. Values greater than 0.75 can be considered 'good'. With different linkage methods, the highest values are given by group averages, followed by simple averages and furthest neighbor. Lowest values are given by Ward's method.

However the user should base his selection not only on the cophonetic correlation coefficient but also on his own knowledge of the data.

Another interactive option is the Kleiner - Hartigan tree which is plotted vertically from the 'root' or 'trunk' (maximum dissimilarity) branching upwards until only one field remains (see Figure 2). The angle between 'branches' is proportional to the similarity. The width of each 'branch' represents the number of variables above it within the 'tree'. The lengths of each 'branch' are fixed and equal. Plotfiles of the individual dendrograms and the Kleiner - Hartigan tree can also be output.

[![](../Images_Commands/Clustr1.gif)](<javascript:void\(0\);>)

Dendogram showing two distinct groupings between CU, CXCU, CXHM, MO, Conductivity and PB, ZN, CO, NI and pH in string sediments.

[![](../Images_Commands/Clustr2.gif)](<javascript:void\(0\);>)

_Kleiner-Hartigan tree of the same data displayed in Figure 1. The decreasing angle between the branches is proportional to the similarity. The width of the branch is proportional to the number of fields above it within the tree. The groupings vary slightly as the linking algorithm used to produce the tree (nearest neighbour) is different._

### Input Files:

* **in** (Undefined):
  Optional raw data input file.
  Required=No

* **matxin** (Undefined):
  Optional matrix input file.
  Required=No

### Output Files:

* **matxfile** (Undefined):
  Output file containing similarity or dissimilarity matrix.
  Required=No

### Fields:

* **sampid** (Any : IN):
  Field containing sample identification or variable identification if a matrix input
  file is used.
  Default=Undefined
  Required=Yes

* **fields** (Undefined : Undefined):
  First field to be used. No fields specified means all.
  Default=Undefined
  Required=No

### Parameters:

* **mattype**:

* **Options** ((0): Product moment correlation matrix. (Similarity Matrix). Note, using):
  default value here, must use default value for ZTRAN.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **ztran**:

* **Options** (0: Z Transformation of data not required to calculate matrix. Only applicable):
  for raw data input.; (1): Z Transformation of data required to calculate matrix.
  Range=0,1
  Values=0,1
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('clustr')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute clustr
print("Running clustr...")
dm_cmd.clustr(
    in_i='t_assays',  # required
    matxin_i='optional',  # required
    matxfile_o='t_clustr_out',  # required
    sampid_f='optional',  # required
    fields_f=['AU'],  # required
    # mattype_p=0,  # optional
    # ztran_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("clustr execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_clustr_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")